# Label Clusters — BEV Bike Labeling

Interactive notebook for hand-labeling obstacle clusters as **bike/scooter** or **other**.

**Workflow:**
1. Run all cells top-to-bottom
2. A grid of bird's-eye-view thumbnails is shown (green = height, blue = point density)
3. Type the **index numbers** of clusters that are parked bikes or scooters
4. Run the save cell to write labels to `data/labels/cluster_labels.csv`
5. Repeat for the next page

A single bike/scooter typically appears as an elongated blob ~1.5–2.0 m long, 0.3–0.6 m wide, at handlebar height (~1.0–1.1 m). Groups of bikes form a row of such blobs.

## Config

In [ ]:
from pathlib import Path

# Labeled point cloud (bgt_labeled output from notebook 2.5)
LAZ_FILE = Path("/Users/minkeverweij/UPSW_neo/data/output/labeled_pointcloud/bgt_labeled_120300_489300.laz")

# Obstacle polygons (output of notebook 3)
OBSTACLES_GEOJSON = Path("/Users/minkeverweij/UPSW_neo/data/output/obstacles/obstacles_2d_120300_488900.geojson")

# Output label file
LABELS_DIR  = Path("data/labels")
LABELS_FILE = LABELS_DIR / "cluster_labels.csv"
LABELS_DIR.mkdir(parents=True, exist_ok=True)

BEV_RESOLUTION  = 0.05   # metres per pixel in the rendered image
BEV_OUTPUT_SIZE = 64     # pixels — covers (64 × 0.05) = 3.2 m × 3.2 m
PAGE_SIZE       = 40     # clusters shown per page
NCOLS           = 8      # columns in the grid

print(f"LAZ:        {LAZ_FILE}")
print(f"Obstacles:  {OBSTACLES_GEOJSON}")
print(f"Labels out: {LABELS_FILE}")

## Load point cloud and compute heights

In [ ]:
import numpy as np
import geopandas as gpd
import laspy

from utils.obstacle_extractor_2d import build_ground_grid, compute_heights, GROUND_LABEL
from utils.bev_renderer import extract_cluster_bev_from_arrays, render_cluster_grid

print("Reading LAZ …")
pc = laspy.read(str(LAZ_FILE))
xyz = np.column_stack([
    np.asarray(pc.x, dtype=np.float64),
    np.asarray(pc.y, dtype=np.float64),
    np.asarray(pc.z, dtype=np.float64),
])
labels_arr = (
    np.asarray(pc.label, dtype=np.int32)
    if "label" in pc.point_format.extra_dimension_names
    else np.zeros(len(xyz), dtype=np.int32)
)
print(f"  {len(xyz):,} points loaded")

print("Building ground grid …")
grid, xmin, ymin = build_ground_grid(xyz, labels_arr)
heights = compute_heights(xyz, grid, xmin, ymin)
print("  Done")

## Load obstacle polygons

In [ ]:
import json
from shapely.geometry import shape

with open(OBSTACLES_GEOJSON) as f:
    gj = json.load(f)

polygons = [shape(feat["geometry"]) for feat in gj["features"]]
print(f"{len(polygons)} obstacle clusters loaded")

## Render BEV images for all clusters

In [ ]:
print("Rendering BEV images …")
bev_images = []
for i, poly in enumerate(polygons):
    img = extract_cluster_bev_from_arrays(
        xyz, heights, poly,
        resolution=BEV_RESOLUTION,
        output_size=BEV_OUTPUT_SIZE,
    )
    bev_images.append(img)

print(f"  {len(bev_images)} images rendered")

## Load existing labels (if any)

In [ ]:
import csv

existing_labels = {}  # cluster_idx -> 'bike' or 'other'
if LABELS_FILE.exists():
    with open(LABELS_FILE) as f:
        for row in csv.DictReader(f):
            # only load labels for the current LAZ / obstacle file
            if row["laz_file"] == str(LAZ_FILE) and row["obstacles_file"] == str(OBSTACLES_GEOJSON):
                existing_labels[int(row["cluster_idx"])] = row["label"]

print(f"Existing labels loaded: {len(existing_labels)} / {len(polygons)} clusters")
labeled_idx  = set(existing_labels.keys())
unlabeled_idx = [i for i in range(len(polygons)) if i not in labeled_idx]
print(f"Remaining to label: {len(unlabeled_idx)}")

## Show a page of unlabeled clusters

Green channel = height, blue channel = density.  
A single parked bike: elongated blob ~1.5–2.0 m long (occupies most of the 3.2 m frame).  
Bike groups: row of blobs side-by-side.  
Run the **next cell** to enter which cluster indices are bikes/scooters.

In [ ]:
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
matplotlib.rcParams["figure.dpi"] = 120

# Clusters to show this page
page_idx = unlabeled_idx[:PAGE_SIZE]
page_imgs = [bev_images[i] for i in page_idx]
areas = [polygons[i].area for i in page_idx]
titles = [f"{i}\n{areas[j]:.1f}m²" for j, i in enumerate(page_idx)]

nrows = (len(page_idx) + NCOLS - 1) // NCOLS
fig, axes = plt.subplots(nrows, NCOLS,
                          figsize=(NCOLS * 1.4, nrows * 1.4),
                          squeeze=False)
for j, ax in enumerate(axes.flat):
    if j >= len(page_idx):
        ax.axis("off")
        continue
    img = page_imgs[j]
    rgb = np.zeros((*img.shape[:2], 3), dtype=np.float32)
    rgb[..., 1] = img[..., 0]   # height  → green
    rgb[..., 2] = img[..., 1]   # density → blue
    ax.imshow(rgb, origin="lower", interpolation="nearest")
    ax.set_xticks([]); ax.set_yticks([])
    ax.set_title(titles[j], fontsize=7, pad=2)

plt.tight_layout(pad=0.3)
plt.show()
print(f"Showing clusters: {page_idx}")


## Enter labels

Edit `bike_indices` below: list the **cluster index numbers** (shown in each thumbnail title) that contain parked bikes or scooters.  
Leave empty `[]` if none on this page are bikes.

In [ ]:
# ← Edit this list with the cluster indices that are bikes/scooters
bike_indices = []

# Build label dict for this page
page_labels = {}
bike_set = set(bike_indices)
for idx in page_idx:
    page_labels[idx] = "bike" if idx in bike_set else "other"

print(f"Page labels set: {len(page_labels)} clusters")
print(f"  bike:  {sum(v == 'bike'  for v in page_labels.values())}")
print(f"  other: {sum(v == 'other' for v in page_labels.values())}")

## Save labels to CSV

In [ ]:
import csv
from pathlib import Path

fieldnames = ["laz_file", "obstacles_file", "cluster_idx", "label", "area_m2"]

# Merge with existing labels (overwrite if re-labeling)
all_labels = dict(existing_labels)   # copy
all_labels.update(page_labels)

# Read any rows from OTHER tiles/files in the CSV
other_rows = []
if LABELS_FILE.exists():
    with open(LABELS_FILE) as f:
        for row in csv.DictReader(f):
            if not (row["laz_file"] == str(LAZ_FILE)
                    and row["obstacles_file"] == str(OBSTACLES_GEOJSON)):
                other_rows.append(row)

with open(LABELS_FILE, "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=fieldnames)
    w.writeheader()
    for row in other_rows:
        w.writerow(row)
    for idx, lbl in sorted(all_labels.items()):
        w.writerow({
            "laz_file":       str(LAZ_FILE),
            "obstacles_file": str(OBSTACLES_GEOJSON),
            "cluster_idx":    idx,
            "label":          lbl,
            "area_m2":        round(polygons[idx].area, 3),
        })

# Advance the page
existing_labels.update(page_labels)
labeled_idx  = set(existing_labels.keys())
unlabeled_idx = [i for i in range(len(polygons)) if i not in labeled_idx]

print(f"Saved → {LABELS_FILE}")
print(f"Total labeled so far: {len(labeled_idx)} / {len(polygons)}")
print(f"Remaining: {len(unlabeled_idx)}")
if unlabeled_idx:
    print("Re-run the 'show-page' cell to see the next page.")
else:
    print("All clusters labeled!")

## Label summary

In [ ]:
import pandas as pd

if LABELS_FILE.exists():
    df = pd.read_csv(LABELS_FILE)
    print(df["label"].value_counts().to_string())
    print(f"\nTotal rows: {len(df)}")
    # Show bike clusters
    bikes = df[df["label"] == "bike"]
    if len(bikes):
        print(f"\nBike clusters (area range): {bikes['area_m2'].min():.2f} – {bikes['area_m2'].max():.2f} m²")